# Insurance Charge Predictor

In [ ]:
%pip show numpy pandas matplotli2b seaborn

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

In [ ]:
data = pd.read_csv("insurance.csv")

In [ ]:
data

## Exploratory Data Analysis

In [ ]:
# first we will check the shape of data
data.shape

In [ ]:
data.head()

In [ ]:
# find out info of data (data types, null values, columns)
data.info()

In [ ]:
# describe the data - find out mean, median and mode of data
# description is of only numerical data
data.describe()

In [ ]:
# check the null values in each column
data.isnull().sum() # .sum() shows the counts of null values

In [ ]:
# we will visulaize the data
# for that, we will find out all numeric data (columns) - continuous vairable
data.columns


In [ ]:
# extracting the numeric columns
numeric_columns = ['age', 'bmi', 'children', 'charges']
for cols in numeric_columns:
    plt.figure(figsize=(6,4)) # means it will be of size (600*400) pixels
    # we will create histogram - best for checking distribution
    # here we can also make kde (kernel distribution estimation)
    sns.histplot(data[cols], kde=True, bins=20)

In [ ]:
# for plotting count plots for columns = children, 
# since there'r no use of histplot for this column
sns.countplot(x = data['children'])

In [ ]:
sns.countplot(x = data['sex'])

In [ ]:
sns.countplot(x = data['smoker'])

In [ ]:
# using box plot, we are checking outliers
for col in numeric_columns:
    plt.figure(figsize=(6,4))
    sns.boxplot(x = data[col])

In [ ]:
# finding out co-relation between data
# heatmap is best for depicting co-relations
# co-relation is only counted for numerical values
plt.figure(figsize=(8,6))
sns.heatmap(data.corr(numeric_only=True), annot=True)
# annotationsn helps in showing value on heatmap itself

## Data cleaning & preprocessing

In [ ]:
# use one more variable to store all my data inside it
data_cleaned = data.copy()

In [ ]:
data_cleaned.head()

In [ ]:
# check rows again in data_cleaned
data_cleaned.shape

In [ ]:
data_cleaned.isnull().sum()

In [ ]:
# remove null values and duplicate 
data_cleaned.dropna(inplace = True)
data_cleaned.drop_duplicates(inplace = True)

In [ ]:
# check updated data_cleaned shape
data_cleaned.shape

In [ ]:
# check data types
data_cleaned.dtypes

In [ ]:
# first of all we need to convert all columns data into numerical valuees
# so, we will check value counts
data_cleaned['sex'].value_counts()

In [ ]:
# converting non-numeric data into numeric
# we will use label-encoding
# we will convert mail -> "0" and female -> "1"
data_cleaned['sex'] = data_cleaned['sex'].map({"male": 0, "female": 1})

In [ ]:
data_cleaned.head()

In [ ]:
data_cleaned['smoker'].value_counts()

In [ ]:
# converting non-numeric data into numeric
# we will use label-encoding
data_cleaned['smoker'] = data_cleaned['smoker'].map({"no": 0, "yes": 1})

In [ ]:
data_cleaned.head()

In [ ]:
# rename column names of 'sex' & 'smoker'
data_cleaned.rename(columns = {
    'sex': 'is_female',
    'smoker': 'is_smoker'}, inplace=True) 

# inplace makes sure that everything actually works

In [ ]:
data_cleaned.head()

In [ ]:
data_cleaned['region'].value_counts()

In [ ]:
# here, there are more than 2 options. so we will use one-hot encoding
# we will be creating another columns for regions separately

data_cleaned = pd.get_dummies(data_cleaned,columns = ['region'],drop_first=True)

# we can have multiple columns also
# drop_first is used -> to discard already present column that is 'region'

In [ ]:
data_cleaned.head()

In [ ]:
# again, we need to convert 'true' & 'false' values into 0 & 1
data_cleaned = data_cleaned.astype(int)

In [ ]:
data_cleaned

## Feature engineering

In [ ]:
# hit and trial by creating or removing new or existing columns
sns.histplot(data['bmi'])

In [ ]:
data_cleaned['bmi_category'] = pd.cut(
    data_cleaned['bmi'],
    bins = [0, 18.5, 24.9, 29.9, float('inf')], # infinity
    labels = ['Underweight', 'Normal', 'Overweight', 'obese']
)

In [ ]:
data_cleaned

In [ ]:
# again we have categorical data and 4 options 
# so we do one-hot encoding
data_cleaned = pd.get_dummies(data_cleaned,columns = ['bmi_category'],drop_first=True)


In [ ]:
data_cleaned

In [ ]:
data_cleaned = data_cleaned.astype(int)

In [ ]:
data_cleaned.head()

## Feature scaling

In [ ]:
# checking columns
data_cleaned.columns

In [ ]:
%pip install scikit-learn

In [ ]:
# only two columns needs feature scaling, 
# since rest are already in 0 & 1
# we will do Standard Scaling -> it is done using skLearn

In [ ]:
data_cleaned.head()

## Feature selection (extraction)

In [ ]:
%pip install scipy

In [ ]:
# we need to extract that columns which is highly co-related to our charges
from scipy.stats import pearsonr

#----------------------
# Pearson Correlation Calculation
#----------------------

# list of features to check against target
selected_features = [
    'age', 'is_female', 'bmi', 'children', 'is_smoker',
    'region_northwest', 'region_southeast', 'region_southwest','bmi_category_Normal', 
    'bmi_category_Overweight', 'bmi_category_obese'
]

correlations = {
    feature: pearsonr(data_cleaned[feature], data_cleaned['charges'])[0]
    for feature in selected_features
}

# converting correlation dictionary into dataframe
correlation_data = pd.DataFrame(list(correlations.items()), columns=['Feature', 'Pearson Correlation'])

# sorting the correlation values -> highest - lowest
correlation_data.sort_values(by='Pearson Correlation', ascending = False)

In [ ]:
# here we will use chi-square test 
# will let u know why we are using it
# but first we do select categorical columns
categorical_features = [
    'is_female', 'is_smoker',
    'region_northwest', 'region_southeast', 'region_southwest',
    'bmi_category_Normal', 'bmi_category_Overweight', 'bmi_category_obese'
]

In [ ]:
from scipy.stats import chi2_contingency # library
import pandas as pd
# we r using chi square contingency table

# we have an alpha value (significance level)
alpha = 0.05

# chi square value -> also called pi value

# creating bins for charges
data_cleaned['charges_bin'] = pd.qcut(data_cleaned['charges'], q=4, labels=False)
chi2_results = {}

# using chi2 contingency
for col in categorical_features:
    contingency = pd.crosstab(data_cleaned[col], data_cleaned['charges_bin'])
    chi2_stat, p_val, _, _ = chi2_contingency(contingency)
    decision = 'Reject Null (Keep Feature)' if p_val < alpha else 'Accept Null (Drop Feature)'
    chi2_results[col] = {
        'chi2_statistic': chi2_stat,
        'p_value': p_val,
        'Decision': decision
    }

chi2_data = pd.DataFrame(chi2_results).T
chi2_data = chi2_data.sort_values(by='p_value')
chi2_data

In [ ]:
final_data = data_cleaned[['age', 'is_female', 'bmi', 'children', 'is_smoker',
                           'charges', 'region_southwest', 'bmi_category_obese'
                          ]]

In [ ]:
final_data

## Train, test and split

In [ ]:
from sklearn.model_selection import train_test_split

# x -> main feature or call it as target feature
# y-> output feature (charges)
X = final_data.drop('charges', axis = 1)
y = final_data['charges']

In [ ]:
# using shift + tab -> will show options to add as param in train_test_split
# use documentation -> scikitlearnn

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
# randome_state -> is not compulsory to use

# train data size 80 (train) and 20 (test)

In [ ]:
from sklearn.preprocessing import StandardScaler

cols = ['age', 'bmi', 'children']

scaler = StandardScaler()

# fit ONLY on training data
X_train[cols] = scaler.fit_transform(X_train[cols])

# transform test data using same scaler
X_test[cols] = scaler.transform(X_test[cols])

## Model Training

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

## Prediction

In [ ]:
# we will find y_predictions to compare with y_test further
# for this, i will use X_train to calculate y_pred

y_pred = model.predict(X_test)

In [ ]:
y_pred

In [ ]:
y_test

## Model Evaluation

# for performance evaluation
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# calculatinh r2

r2 = r2_score(y_test, y_pred)
# MAE
mae = mean_absolute_error(y_test, y_pred)

# MSE
mse = mean_squared_error(y_test, y_pred)

# RMSE
rmse = np.sqrt(mse)

# calculating adjusted r2
# n = total no of columns

n = X_test.shape[0] # -> it gives you no of rows
p = X_test.shape[1] # -> gives no of cols
adjusted_r2 = 1 - ((1-r2) * (n-1) / (n-p-1))
adjusted_r2

print("r2 :", r2)
print("adjusted_2 :", adjusted_r2)
print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)

- Now, we will try out other algorithms one by one.
- Then will calculate each metrics respectively
- Finally, we will compare all results.

# Model Training - All at once

In [ ]:
# importing regression models

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

# metrics
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import numpy as np
import pandas as pd

# all models

models = {
    "Decision Tree": DecisionTreeRegressor(random_state=42),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    ),

    "SVR": SVR(),

    "XGBoost": XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
}


results = []

# training and evaluation of models

for name, model in models.items():

    # train model
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # metrics
    r2 = r2_score(y_test, y_pred)

    mae = mean_absolute_error(y_test, y_pred)

    mse = mean_squared_error(y_test, y_pred)

    rmse = np.sqrt(mse)

    # adjusted r2
    n = X_test.shape[0]
    p = X_test.shape[1]

    adjusted_r2 = 1 - ((1-r2)*(n-1)/(n-p-1))

    # append results
    results.append({
        "Model": name,
        "R2 Score": r2,
        "Adjusted R2": adjusted_r2,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse
    })

# result dataframe

results_df = pd.DataFrame(results)

# sorting by highest r2
results_df = results_df.sort_values(
    by="R2 Score",
    ascending=False
)

print(results_df)

### Separately - XGboost model & training again

In [ ]:
# ------------------------------------------
# Final Original XGBoost Model
# ------------------------------------------

original_xgb_model = XGBRegressor(

    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# training model
original_xgb_model.fit(X_train, y_train)

print("Original XGBoost Model Trained Successfully")

In [ ]:
original_y_pred = model.predict(X_test)
original_xgb_modelr2 = r2_score(y_test, original_y_pred)
original_xgb_modelr2

## Performing Hyperparameter tuning for Xgboost model

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
xgb_model = XGBRegressor(
    random_state=42
)

# using RandomizedSearchCV and will try multiple combinations automatically

param_grid = {

    # number of trees
    'n_estimators': [100, 200, 300],

    # depth of each tree
    'max_depth': [3, 4, 5, 6],

    # learning speed
    'learning_rate': [0.01, 0.05, 0.1],

    # percentage of rows used per tree
    'subsample': [0.8, 0.9, 1.0],

    # percentage of columns used per tree
    'colsample_bytree': [0.8, 0.9, 1.0]
}

### Randomized Search

- RandomizedSearchCV tries random parameter combinations
- and finds best performing model


In [ ]:
random_search = RandomizedSearchCV(

    estimator=xgb_model,

    param_distributions=param_grid,

    # number of parameter combinations to try
    n_iter=10,

    # evaluation metric
    scoring='r2',

    # cross validation folds
    cv=5,

    # verbose output
    verbose=2,

    # use all CPU cores
    n_jobs=-1,

    # reproducibility
    random_state=42
)

In [ ]:
# Implementing randomized search
random_search.fit(X_train, y_train)

In [ ]:
print("Best Parameters:")
print(random_search.best_params_)

print("\nBest R2 Score:")
print(random_search.best_score_)

### Creating final tuned XGBoost model


In [ ]:
final_xgb_model = XGBRegressor(

    n_estimators=random_search.best_params_['n_estimators'],

    max_depth=random_search.best_params_['max_depth'],

    learning_rate=random_search.best_params_['learning_rate'],

    subsample=random_search.best_params_['subsample'],

    colsample_bytree=random_search.best_params_['colsample_bytree'],

    random_state=42
)

final_xgb_model.fit(X_train, y_train)

print("Final Tuned XGBoost Model Trained Successfully")

## Prediction + Evaluation using Final Tuned XGBoost Model

In [ ]:
y_pred_final = final_xgb_model.predict(X_test)

# r2 score
r2_final = r2_score(y_test, y_pred_final)

# MAE
mae_final = mean_absolute_error(y_test, y_pred_final)

# MSE
mse_final = mean_squared_error(y_test, y_pred_final)

# RMSE 
rmse_final = np.sqrt(mse_final)

# adjusted R2
n = X_test.shape[0]   # number of rows
p = X_test.shape[1]   # number of features

adjusted_r2_final = 1 - (
    ((1 - r2_final) * (n - 1)) / (n - p - 1)
)

In [ ]:
# final results
print("Final Tuned XGBoost Results")
print("-" * 40)

print("R2 Score        :", r2_final)
print("Adjusted R2     :", adjusted_r2_final)
print("MAE             :", mae_final)
print("MSE             :", mse_final)
print("RMSE            :", rmse_final)

# Feature Importance Plotting

In [ ]:
# Extracting feature importance scores
importance_scores = original_xgb_model.feature_importances_

# creating dataframes
feature_importance_df = pd.DataFrame({

    'Feature': X_train.columns,

    'Importance': importance_scores
})

# Sorting feature importance
feature_importance_df = feature_importance_df.sort_values(

    by='Importance',

    ascending=False
)

# feature importance table
print(feature_importance_df)

# plotting
plt.figure(figsize=(10,6))

plt.barh(
    feature_importance_df['Feature'],
    feature_importance_df['Importance']
)

# highest importance on top
plt.gca().invert_yaxis()

# labels and title
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.title("Feature Importance - XGBoost")

plt.show()

# Save Model

In [ ]:
import joblib

# save model
joblib.dump(original_xgb_model, "insurance_xgb_model.pkl")

print("Model saved successfully as insurance_xgb_model.pkl")

In [ ]:
# saving features
# saving column structure for inference
joblib.dump(X_train.columns, "model_features.pkl")

print("Feature list saved successfully")